# 17-class pilot: Eileen's shards only

A practice run before `combined/` exists. It trains one model on the `analog`, `multicarrier` and `apsk` shards to see how those 17 classes separate.

**This is not seed 5.** It has 17 classes and no mixtures, so its scores will run higher than the merged model's. Don't report them, and don't save this model to `members/`.

Before you start: **Runtime → Change runtime type → GPU**.


## 1. Get the code


In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-17class-pilot https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pip install -q pyyaml h5py gdown


## 2. Check the GPU

This must say `CUDA: True`. If it doesn't, change the runtime type and start again.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Turn on the GPU: Runtime -> Change runtime type'


## 3. Save results to your Drive

Checkpoints go here every epoch, so a dropped session can carry on.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nexa_pilot_17class'
!mkdir -p "$OUT"


## 4. Download the shards

About 1.1 GB from `processed_data`. The next step checks every file against its manifest, so a bad download stops the run.


In [ ]:
FOLDERS = {
    'analog': '1oz2amRn65zqJ_JWymViaPhZ0wANdFLuZ',
    'multicarrier': '16FsXoJ4yc--pvm3yv89vd8RhTPh3VR2o',
    'apsk': '1uidz9R9NUz91mYe9ko1o8gBzX5Z8sedr',
}
for group, folder in FOLDERS.items():
    !gdown --folder "https://drive.google.com/drive/folders/{folder}" -O "/content/shards/{group}"
!ls -la /content/shards/*


## 5. Train

Check first, a minute or two on a small sample. If this fails, stop and fix it.


In [ ]:
!python scripts/pilot_train_shards.py --shards /content/shards --out /content/pilot_check --limit 5000 --epochs 1


The real run. Each epoch prints a line. If the session drops, run cells 1 to 4 again and then this one: it resumes from your Drive.


In [ ]:
!python scripts/pilot_train_shards.py --shards /content/shards --out "$OUT"


## 6. Results

Weakest classes first. Send `scorecard.json` from your Drive to Claude to go through it.


In [ ]:
import json
card = json.load(open(f'{OUT}/scorecard.json'))
print('macro recall', round(card['macro_recall'], 3), 'on', card['n_windows'], 'test windows')
for name, c in sorted(card['per_class'].items(), key=lambda kv: kv[1]['recall']):
    print(f"{name:<12} recall {c['recall']:.3f}  false alarm {c['false_alarm']:.3f}  {list(c['confused_with'])[:2]}")
